In [150]:
import bs4
from langsmith import Client
from langchain_community.document_loaders import WebBaseLoader
from langchain_ollama import ChatOllama
from langchain_ollama import OllamaEmbeddings
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_core.prompts import ChatPromptTemplate
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.messages.base import BaseMessage
from langchain_community.docstore.document import Document

# from langchain.schema.runnable import RunnablePassthrough
from langchain_core.runnables import RunnablePassthrough, RunnableLambda

# from langchain.schema.output_parser import StrOutputParser
from langchain_core.output_parsers import StrOutputParser


In [151]:
client = Client()

In [152]:
llm = ChatOllama(
    model="deepseek-r1:1.5b",
    temperature=0,
)


In [153]:
embeddings = OllamaEmbeddings(model="qwen3-embedding:0.6b")

In [154]:
vector_store = InMemoryVectorStore(embeddings)

In [156]:
docs = [
    Document(page_content="USA president say about RUSSIA a lot of good things. For example: good coal and gas", metadata={"source": "USA records"})
]

In [157]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,  # chunk size (characters)
    chunk_overlap=200,  # chunk overlap (characters)
    add_start_index=True,  # track index in original document
)

all_splits = text_splitter.split_documents(docs)

print(f"Split blog post into {len(all_splits)} sub-documents.")


Split blog post into 1 sub-documents.


In [158]:
document_ids = vector_store.add_documents(documents=all_splits)
len(document_ids), document_ids[:5]


(1, ['d110ad66-828e-4580-8bc5-b06357e34298'])

In [159]:
retriever = vector_store.as_retriever()

In [160]:
template = """You are an assistant for question-answering tasks. 
Use the following pieces of retrieved context to answer the question. 
If you don't know the answer, just say that you don't know. 
Use three sentences maximum and keep the answer concise.
Question: {question} 
Context: {context} 
Answer:
"""
prompt_template = ChatPromptTemplate.from_template(template)
prompt_template

ChatPromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template="You are an assistant for question-answering tasks. \nUse the following pieces of retrieved context to answer the question. \nIf you don't know the answer, just say that you don't know. \nUse three sentences maximum and keep the answer concise.\nQuestion: {question} \nContext: {context} \nAnswer:\n"), additional_kwargs={})])

In [161]:
question = (
    "what say USA president about RUSSIA?\n\n"
    "Once you get the answer, look up common extensions of that method."
)

In [162]:
llm.invoke(question)

AIMessage(content='The U.S. president would typically respond by stating their name followed by "Russia" as the country they represent. For example:\n\n"The President of the United States is [president\'s name]."\n\nThis response is concise and formal, addressing the question about Russia effectively.', additional_kwargs={}, response_metadata={'model': 'deepseek-r1:1.5b', 'created_at': '2026-02-24T14:59:08.580436334Z', 'done': True, 'done_reason': 'stop', 'total_duration': 3513301476, 'load_duration': 87559558, 'prompt_eval_count': 26, 'prompt_eval_duration': 190752375, 'eval_count': 307, 'eval_duration': 2812841465, 'logprobs': None, 'model_name': 'deepseek-r1:1.5b', 'model_provider': 'ollama'}, id='lc_run--019c9029-32e9-7dc0-bd47-505df79efe07-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 26, 'output_tokens': 307, 'total_tokens': 333})

In [163]:


rag_chain = (
    {
        "context": retriever,
        "question": RunnablePassthrough()
    }
    | prompt_template
    | llm
)

In [ ]:
rag_chain.invoke(question)

AIMessage(content='The USA president has mentioned a lot of good things about Russia, including its coal and gas resources. For example, it says "good coal and gas."', additional_kwargs={}, response_metadata={'model': 'deepseek-r1:1.5b', 'created_at': '2026-02-24T14:59:10.097690381Z', 'done': True, 'done_reason': 'stop', 'total_duration': 1403320466, 'load_duration': 68814089, 'prompt_eval_count': 164, 'prompt_eval_duration': 39084294, 'eval_count': 141, 'eval_duration': 1148495108, 'logprobs': None, 'model_name': 'deepseek-r1:1.5b', 'model_provider': 'ollama'}, id='lc_run--019c9029-4115-71e3-be26-cb4a8c19e5dc-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 164, 'output_tokens': 141, 'total_tokens': 305})